<a href="https://colab.research.google.com/github/mk654/SML_PG60/blob/main/COMP90051_ProjectGroup60_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **COMP90051 Group Project → Code**

| Project Group 60 |         |
|------------------|---------|
| Lachlan Fox      | 649622  |
| Songhao Guo      | 1542657 |
| Amelia King      | 1175861 |


github → https://github.com/mk654/SML_PG60


In [ ]:
# RUN FIRST
from scipy.io import loadmat
import pandas as pd
from pathlib import Path
import numpy as np
import requests
import os

# load data from gitrepo

url = "https://raw.githubusercontent.com/mk654/SML_PG60/main/influenza_outbreak_dataset.mat"
#url = "https://raw.githubusercontent.com/mk654/SML_PG60/main/data/influenza_outbreak_dataset.mat"



mat_path = Path("influenza_outbreak_dataset.mat")

# Use HEAD request to get file size if available
expected_file_size = None
try:
    head_response = requests.head(url)
    head_response.raise_for_status()
    if 'Content-Length' in head_response.headers:
        expected_file_size = int(head_response.headers['Content-Length'])
        print(f"Expected file size (from Content-Length header): {expected_file_size} bytes")
except requests.exceptions.RequestException as e:
    print(f"Could not get Content-Length via HEAD request: {e}")

r = requests.get(url, stream=True)
r.raise_for_status()

downloaded_size = 0
with open(mat_path, "wb") as f:
    for chunk in r.iter_content(chunk_size=8192):
        if chunk:  # filter out keep-alive new chunks
            f.write(chunk)
            downloaded_size += len(chunk)

# Check the size of the downloaded file on disk
if mat_path.exists():
    file_size_on_disk = os.path.getsize(mat_path)
    print(f"Downloaded file size on disk: {file_size_on_disk} bytes")
    if expected_file_size and file_size_on_disk != expected_file_size:
        print(f"WARNING: Downloaded file size ({file_size_on_disk} bytes) does not match expected size ({expected_file_size} bytes).")
    elif not expected_file_size:
        print("Could not retrieve expected file size from Content-Length header.")
else:
    print(f"File '{mat_path}' does not exist after download attempt.")

data = loadmat(mat_path)

Expected file size (from Content-Length header): 4772170 bytes
Downloaded file size on disk: 4772170 bytes


### 0.a) inspecting data
*results*
| name     | outer dtype | outer shape | inner type | inner shape |
|----------|-------------|-------------|------------|-------------|
| X train  | object      | (1, 48)     | csc_matrix | (1095, 545) |
| X test   | object      | (1, 48)     | csc_matrix | (485, 545)  |
| y train  | object      | (1, 48)     | ndarray    | (1095, 1)   |
| y test   | object      | (1, 48)     | ndarray    | (485, 1)    |
| locs     | object      | (1, 48)     | ndarray    | (1,)        |
| keywords | object      | (1, 525)    | ndarray    | (1,)        |

in sum:
- 48 training feature matrices, one per location
- 48 testing feature matrices, one per location
- 48 training label vectors
- 48 testing label vectors
- 48 location names/IDs
- 545 keyword feature names


*Code*


```
rows = []
for name in ["flu_X_tr", "flu_X_te", "flu_Y_tr", "flu_Y_te", "flu_locs", "flu_keywords"]:
    value = data[name]
    try:
        first = value[0, 0]
        first_type = type(first).__name__
        first_shape = getattr(first, 'shape', None)
    except:
        first_shape = None
        first_type = None
    rows.append({
        "name": name,
        "outer_dtype": value.dtype,
        "outer_shape": value.shape,
        "inner_type": first_type,
        "inner_shape": first_shape
    })
df_summary = pd.DataFrame(rows)
print(df_summary)
```



In [ ]:
# delete this cell before submission!!

rows = []
for name in ["flu_X_tr", "flu_X_te", "flu_Y_tr", "flu_Y_te", "flu_locs", "flu_keywords"]:
    value = data[name]
    try:
        first = value[0, 0]
        first_type = type(first).__name__
        first_shape = getattr(first, 'shape', None)
    except:
        first_shape = None
        first_type = None
    rows.append({
        "name": name,
        "outer_dtype": value.dtype,
        "outer_shape": value.shape,
        "inner_type": first_type,
        "inner_shape": first_shape
    })
df_summary = pd.DataFrame(rows)
print(df_summary)

           name outer_dtype outer_shape inner_type  inner_shape
0      flu_X_tr      object     (1, 48)  csc_array  (1095, 545)
1      flu_X_te      object     (1, 48)  csc_array   (485, 545)
2      flu_Y_tr      object     (1, 48)    ndarray    (1095, 1)
3      flu_Y_te      object     (1, 48)    ndarray     (485, 1)
4      flu_locs      object     (1, 48)    ndarray         (1,)
5  flu_keywords      object    (1, 525)    ndarray         (1,)


### 0.b) convert .mat to .csv